# 12 — Rotating BTZ: horizons, ergoregion, thermodynamics, rotating Strominger (v1.3 patch)

**Spacetime Lab v1.3 — the full two-parameter BTZ family**

Phase 8 (v0.8.0) shipped the non-rotating BTZ: a 2+1D black hole characterised by a single horizon radius $r_+$. The Strominger 1998 derivation of $S_{BH}$ from the boundary CFT Cardy formula matched bit-exactly. But non-rotating BTZ misses some of the most interesting physics: the **ergoregion**, the **first law** coupling temperature and angular velocity, and the **chiral split** of the boundary CFT into asymmetric left/right Virasoro towers.

This v1.3 patch ships the full $(M, J)$ family.

## What's new vs Phase 8

| | Phase 8 (`btz.py`) | v1.3 (`btz_rotating.py`) |
|---|---|---|
| Parameters | $r_+$ | $(M, J)$, with two horizons $r_\pm$ |
| Rotation | none | $J$ up to extremal $J = ML$ |
| Ergoregion | trivial (point) | $r_+ < r < \sqrt{8 G_N L^2 M}$ |
| Thermodynamics | $T_H$, $S_{BH}$ | $+ \Omega_H$, first law, Smarr 2+1D |
| Cardy split | symmetric $L_0 = \bar L_0$ | asymmetric $L_0 \ne \bar L_0$ |
| Strominger match | $r_+/(8 G_N) + r_+/(8 G_N) = r_+/(2 G_N)/2$ | $(r_+ + r_-)/(4 G_N) + (r_+ - r_-)/(4 G_N) = r_+/(2 G_N)$ |

Both pass bit-exact gates.

## 1. Horizon structure

Solving $f(r) = 0$ for the metric

$$ds^2 = -f(r)\,dt^2 + \frac{dr^2}{f(r)} + r^2\left(d\phi - \frac{r_+ r_-}{r^2 L}\,dt\right)^2,\qquad f(r) = \frac{(r^2 - r_+^2)(r^2 - r_-^2)}{r^2 L^2}$$

gives two real positive roots $r_\pm$ whenever $|J| \le ML$. The closed form:

$$r_\pm^2 = 4 G_N L^2 M \left(1 \pm \sqrt{1 - (J/(ML))^2}\right),$$

equivalently $r_+^2 + r_-^2 = 8 G_N L^2 M$ and $r_+ r_- = 4 G_N L\,|J|$.

**Extremality**: at $|J| = ML$ the two horizons merge ($r_+ = r_-$) and the Hawking temperature vanishes. Beyond $|J| > ML$ there is no horizon — a naked conical singularity.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge, Circle

from spacetime_lab.holography import (
    rotating_btz_horizons,
    rotating_btz_hawking_temperature,
    rotating_btz_angular_velocity,
    rotating_btz_entropy,
    rotating_btz_ergoregion_bounds,
    strominger_rotating_btz_cardy,
    verify_first_law,
    verify_smarr_2plus1,
    verify_rotating_btz_thermodynamics,
    extremal_bound_J,
)

# Canonical reference point
M, J, L, G_N = 1.0, 0.5, 1.0, 1.0
r_p, r_m = rotating_btz_horizons(M, J, L, G_N=G_N)
print(f'M = {M}, J = {J}, L = {L}, G_N = {G_N}')
print(f'r_+ = {r_p:.10f}')
print(f'r_- = {r_m:.10f}')
print(f'r_+^2 + r_-^2 = {r_p**2 + r_m**2:.10f}  (expected 8 G_N L^2 M = {8*G_N*L**2*M:.10f})')
print(f'r_+ r_-       = {r_p*r_m:.10f}  (expected 4 G_N L |J| = {4*G_N*L*abs(J):.10f})')
print(f'|J|/(ML)      = {abs(J)/(M*L):.10f}  (extremal = 1.0)')

## 2. Phase diagram $(M, J)$

The allowed region is the triangle $|J| \le ML$. The line $J = \pm ML$ is extremal (zero temperature). Inside the region, curves of constant $T_H$ sweep from the $M$-axis (non-rotating, highest $T$ at given $M$) toward the extremal boundary.

In [ ]:
Ms = np.linspace(0.01, 3.0, 80)
Js = np.linspace(-3.0, 3.0, 160)
M_grid, J_grid = np.meshgrid(Ms, Js, indexing='xy')
T_grid = np.full_like(M_grid, np.nan)

for i in range(M_grid.shape[0]):
    for j in range(M_grid.shape[1]):
        M_ij = M_grid[i, j]
        J_ij = J_grid[i, j]
        if abs(J_ij) <= M_ij * L:
            T_grid[i, j] = rotating_btz_hawking_temperature(M_ij, J_ij, L)

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(Ms, Js, T_grid, levels=20, cmap='viridis')
cb = plt.colorbar(cs, ax=ax, label=r'$T_H$')

# Extremal lines J = ±ML
ax.plot(Ms, +Ms * L, color='red', lw=2, label=r'extremal $J = +ML$ ($T_H = 0$)')
ax.plot(Ms, -Ms * L, color='red', lw=2, ls='--', label=r'extremal $J = -ML$')

ax.set_xlabel(r'$M$', fontsize=12)
ax.set_ylabel(r'$J$', fontsize=12)
ax.set_title('Rotating BTZ phase diagram: allowed region $|J| \\le ML$, colour = $T_H$')
ax.legend(loc='upper right')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 3. Approach to extremality: $T_H \to 0$, $\Omega_H \to 1/L$

At fixed $M$, sweep $J$ from $0$ to $ML$. $T_H$ drops to zero; $\Omega_H$ rises to the extremal value $1/L$.

In [ ]:
M_fixed = 1.0
J_max = extremal_bound_J(M_fixed, L)
Js_sweep = np.linspace(0.0, 0.9999 * J_max, 100)
Ts = [rotating_btz_hawking_temperature(M_fixed, J, L) for J in Js_sweep]
Oms = [rotating_btz_angular_velocity(M_fixed, J, L) for J in Js_sweep]
Ss = [rotating_btz_entropy(M_fixed, J, L) for J in Js_sweep]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].plot(Js_sweep / J_max, Ts, lw=2.2, color='#d62728')
axes[0].set_xlabel(r'$J / J_\mathrm{max}$')
axes[0].set_ylabel(r'$T_H$')
axes[0].set_title(r'Temperature $\to 0$ at extremality')
axes[0].axvline(1.0, color='gray', ls=':')
axes[0].grid(alpha=0.3)

axes[1].plot(Js_sweep / J_max, Oms, lw=2.2, color='#1f77b4')
axes[1].axhline(1.0 / L, color='gray', ls='--', label=r'$\Omega_H^\mathrm{ext} = 1/L$')
axes[1].set_xlabel(r'$J / J_\mathrm{max}$')
axes[1].set_ylabel(r'$\Omega_H$')
axes[1].set_title(r'Horizon angular velocity grows to $1/L$')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(Js_sweep / J_max, Ss, lw=2.2, color='#2ca02c')
axes[2].set_xlabel(r'$J / J_\mathrm{max}$')
axes[2].set_ylabel(r'$S_{BH}$')
axes[2].set_title(r'Entropy: $r_+$ grows modestly, $r_-$ catches up')
axes[2].grid(alpha=0.3)

plt.suptitle(f'Approach to extremality at $M = {M_fixed}$', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Ergoregion — where static observers cannot exist

In rotating BTZ, the Killing vector $\partial_t$ becomes spacelike in an annulus between the outer horizon $r_+$ and the static limit $r_\text{erg} = \sqrt{r_+^2 + r_-^2} = \sqrt{8 G_N L^2 M}$.

Inside the ergoregion, no observer can remain at rest relative to infinity — frame-dragging forces them to co-rotate with the BH. This is the 2+1D analogue of the Kerr ergoregion, and the setting for Penrose's energy extraction mechanism (Penrose process).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

# Choose a rotating BTZ well inside the allowed region
M_plot, J_plot = 1.0, 0.7
r_p, r_m = rotating_btz_horizons(M_plot, J_plot, L)
r_in, r_out = rotating_btz_ergoregion_bounds(M_plot, J_plot, L)

# Ergoregion as shaded annulus
theta = np.linspace(0, 2*np.pi, 200)
for radius, color, label in [(r_m, '#888888', fr'inner horizon $r_-={r_m:.3f}$'),
                              (r_p, '#000000', fr'outer horizon $r_+={r_p:.3f}$'),
                              (r_out, '#d62728', fr'static limit $r_\mathrm{{erg}}={r_out:.3f}$')]:
    ax.plot(radius * np.cos(theta), radius * np.sin(theta),
            color=color, lw=2.2, label=label)

# Shade ergoregion
ergo_patch = Wedge((0, 0), r_out, 0, 360, width=(r_out - r_p), facecolor='#d62728',
                   alpha=0.22, label='ergoregion')
ax.add_patch(ergo_patch)

# Shade interior (between r_- and r_+)
interior_patch = Wedge((0, 0), r_p, 0, 360, width=(r_p - r_m), facecolor='#333333',
                       alpha=0.25, label='BH interior')
ax.add_patch(interior_patch)

ax.set_aspect('equal')
ax.set_xlim(-1.1 * r_out, 1.1 * r_out)
ax.set_ylim(-1.1 * r_out, 1.1 * r_out)
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.set_title(fr'Rotating BTZ with $M={M_plot}$, $J={J_plot}$: ergoregion (red), interior (grey)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. First law and Smarr — numerical verification

$$dM = T_H\,dS + \Omega_H\,dJ$$
$$M = \tfrac{1}{2} T_H S + \Omega_H J\qquad\text{(Smarr, 2+1D)}$$

We compute $(\partial M/\partial S)_J$ and $(\partial M/\partial J)_S$ by symmetric finite differences and compare to the closed-form $T_H$, $\Omega_H$. Smarr is checked algebraically.

In [ ]:
fl = verify_first_law(M_plot, J_plot, L)
sm = verify_smarr_2plus1(M_plot, J_plot, L)

print('First law:  dM = T_H dS + Ω_H dJ')
print(f'  T_H (closed)       = {fl["T_H"]:.12f}')
print(f'  dM/dS (numerical)  = {fl["dM_dS_numerical"]:.12f}')
print(f'  rel residual       = {fl["dM_dS_rel_residual"]:.2e}')
print()
print(f'  Ω_H (closed)       = {fl["Omega_H"]:.12f}')
print(f'  dM/dJ (numerical)  = {fl["dM_dJ_numerical"]:.12f}')
print(f'  rel residual       = {fl["dM_dJ_rel_residual"]:.2e}')
print()
print(f'Smarr 2+1D: M = 1/2 T_H S + Ω_H J')
print(f'  M                  = {sm["M"]:.12f}')
print(f'  1/2 T S + Ω J      = {sm["smarr_rhs"]:.12f}')
print(f'  residual           = {sm["residual"]:.2e}')

## 6. Rotating Strominger-Cardy — the headline match

$$S_{BH} = \frac{\pi r_+}{2 G_N} \stackrel{?}{=} 2\pi\sqrt{\frac{c\,L_0}{6}} + 2\pi\sqrt{\frac{c\,\bar L_0}{6}} = S_\mathrm{Cardy}^L + S_\mathrm{Cardy}^R$$

with asymmetric Virasoro weights $L_0 = (LM + |J|)/2$, $\bar L_0 = (LM - |J|)/2$ and Brown-Henneaux $c = 3L/(2 G_N)$.

Algebraic identity: $S_\mathrm{Cardy}^L = \pi(r_+ + r_-)/(4 G_N)$, $S_\mathrm{Cardy}^R = \pi(r_+ - r_-)/(4 G_N)$. The sum is $\pi r_+/(2 G_N) = S_{BH}$, bit-exactly. The $r_-$ pieces cancel.

In [ ]:
diag = strominger_rotating_btz_cardy(M_plot, J_plot, L)

print('Rotating Strominger-Cardy derivation:')
print(f'  L_0 (left)         = (LM + |J|)/2 = {diag["l_0_left"]:.12f}')
print(f'  L̄_0 (right)        = (LM - |J|)/2 = {diag["l_0_right"]:.12f}')
print(f'  c (Brown-Henneaux) = 3L/(2 G_N)   = {diag["c"]:.12f}')
print()
print(f'  S_Cardy_left       = {diag["s_cardy_left"]:.12f}')
print(f'  S_Cardy_right      = {diag["s_cardy_right"]:.12f}')
print(f'  sum                = {diag["s_cardy_total"]:.12f}')
print(f'  S_BH (bulk)        = {diag["s_bh"]:.12f}')
print(f'  residual           = {diag["residual"]:.2e}')
print()

# Side-by-side bar plot
fig, ax = plt.subplots(figsize=(7, 4))
labels = ['$S_\\mathrm{Cardy}^L$', '$S_\\mathrm{Cardy}^R$', 'sum', '$S_{BH}$']
values = [diag['s_cardy_left'], diag['s_cardy_right'], diag['s_cardy_total'], diag['s_bh']]
colors = ['#1f77b4', '#2ca02c', '#17becf', '#000000']
bars = ax.bar(labels, values, color=colors, alpha=0.82)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.4f}',
            ha='center', va='bottom', fontsize=10)
ax.set_ylabel('entropy')
ax.set_title(fr'Rotating Strominger-Cardy match at $(M, J) = ({M_plot}, {J_plot})$')
plt.tight_layout()
plt.show()

## 7. Closing gate — every claim bit-exact

In [ ]:
diag = verify_rotating_btz_thermodynamics(M_plot, J_plot, L)
print('=== v1.3 rotating BTZ gate ===')
for k, v in diag.items():
    print(f'  {k:.<40} {v}')
print()

assert diag['mass_roundtrip_residual']       < 1e-12
assert diag['angular_momentum_roundtrip_residual'] < 1e-12
assert diag['first_law_dM_dS_rel_residual']  < 1e-6
assert diag['first_law_dM_dJ_rel_residual']  < 1e-6
assert diag['smarr_residual']                < 1e-12
assert diag['strominger_cardy_residual']     < 1e-12

print('ALL GATES PASS — v1.3 is good to ship.')

## Scope notes

**v1.3 ships:**
- `rotating_btz_horizons`, `rotating_btz_mass_from_horizons`, `rotating_btz_angular_momentum_from_horizons`
- `rotating_btz_hawking_temperature`, `rotating_btz_angular_velocity`, `rotating_btz_entropy`
- `rotating_btz_ergoregion_bounds`
- `extremal_bound_J`, `is_extremal`
- `strominger_rotating_btz_cardy` — the asymmetric chiral CFT match
- `verify_first_law`, `verify_smarr_2plus1`, `verify_rotating_btz_thermodynamics`
- Notebook 12 with phase diagram, approach-to-extremality curves, ergoregion cartoon, Strominger bar plot, closing gate

**Deferred:**
- **Geodesics / superradiance in rotating BTZ** — out of scope; the v1.0 RT machinery in `btz.py` only handles the non-rotating limit
- **Chiral CFT with $c_L \ne c_R$** — Einstein gravity has $c_L = c_R$; gravitational Chern-Simons would split them, not scoped here
- **Exact extremal limit $J = ML$** — singular by $r_+ = r_-$; handled as "near-extremal" with warnings, exact extremal state is analytically subtle

The rotating Strominger-Cardy match is the most satisfying physics in this patch: the CFT splits into two asymmetric chiralities, each contributes a non-trivial Cardy entropy depending on $r_\pm$, and the $r_-$ pieces cancel exactly to leave $\pi r_+/(2 G_N)$ — the bulk Bekenstein-Hawking value. Every piece of the AdS/CFT dictionary conspires to make this work.